# colour rays2d

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elma16/beamax/blob/main/examples/rays/colour_rays2d.ipynb)

> Select **Runtime → Change runtime type → GPU or TPU** in Colab to demonstrate the hardware-acceleration story.


In [ ]:
# Install beamax for Google Colab. Safe to skip when running locally.
%%capture
%pip install --quiet "beamax @ git+https://github.com/elma16/beamax.git"


In [ ]:
import jax.numpy as jnp
import jax
import diffrax
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

from beamax import utils
from beamax.gb import gb_utils, gb_solvers
from beamax.plotter import use_beamax_style


ROOT_DIR = utils.detect_root()
PLOT_DIR = Path(ROOT_DIR / "plots")
DATA_DIR = Path(ROOT_DIR / "data")
PROF_DIR = Path(ROOT_DIR / "profiler")
PLOT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)
PROF_DIR.mkdir(exist_ok=True)


use_beamax_style()

jax.config.update("jax_enable_x64", True)

## Trapping Rays

In [ ]:
# Domain setup
d = 2
N = jnp.array([128, 128])
xmax = jnp.array([1.5, 1.5])
dx = xmax / N
x_linspace = [jnp.linspace(0, xmax[i], N[i]) for i in range(d)]

XY = jnp.stack(
    jnp.meshgrid(*x_linspace, indexing="ij"),
    axis=-1,
)


def c_maxwell_fisheye(x, c0=1.0, center=jnp.array([0.75, 0.75]), R=0.35):
    r2 = jnp.sum((x - center) ** 2, axis=-1)
    return c0 * (1.0 + r2 / (R**2))


def wrap_to_pi(a):
    return (a + jnp.pi) % (2 * jnp.pi) - jnp.pi  # → (-π, π]


def _pi_tick_labels(ticks):
    """Return mathtext labels like -π, -π/2, 0, π/2, ... for the given tick values."""
    labels = []
    for t in ticks:
        # Snap near integer multiples to avoid ugly floats
        # k = np.round(t / PI, 6)
        # Try common fractions
        frac = None
        for den in (1, 2, 3, 4, 6, 8, 12):
            num = np.round(den * t / np.pi)
            if np.isclose(t, (num * np.pi) / den, rtol=0, atol=1e-8):
                frac = (int(num), den)
                break
        if frac is None:
            # fallback to numeric with π factor
            labels.append(rf"{t / np.pi:.2f}$\pi$")
            continue
        num, den = frac
        if num == 0:
            labels.append("0")
            continue
        sign = "-" if num < 0 else ""
        a = abs(num)
        if den == 1:
            core = r"\pi" if a == 1 else rf"{a}\pi"
        else:
            if a == 1:
                core = rf"\frac{{\pi}}{{{den}}}"
            else:
                core = rf"\frac{{{a}\pi}}{{{den}}}"
        labels.append(f"${sign}{core}$")
    return labels


def set_colorbar_pi_ticks(cbar, vmin, vmax):
    """Set reasonable π ticks for colorbar in either [-π,π] or [0,2π]."""
    rng = (float(vmin), float(vmax))
    if rng[0] >= -1e-6 and rng[1] <= 2 * np.pi + 1e-6:
        # [0, 2π] style
        base = np.array([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi], dtype=float)
    else:
        # default to [-π, π]
        base = np.array([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi], dtype=float)
    ticks = base[(base >= rng[0] - 1e-9) & (base <= rng[1] + 1e-9)]
    if ticks.size < 3:  # extend if the range is odd
        extra = np.linspace(rng[0], rng[1], 5)
        ticks = np.unique(np.concatenate([ticks, extra]))
    cbar.set_ticks(ticks)
    cbar.set_ticklabels(_pi_tick_labels(ticks))


# Ray setup - fan out from single point
n_rays = 90  # Number of rays to shoot
center = jnp.array([xmax[0] / 2, xmax[1] / 2])
r0 = 0.18
thetas = jnp.linspace(0, 2 * jnp.pi, n_rays, endpoint=False)
# per-ray start points (shape: [n_rays, 2])
start_point = center + r0 * jnp.stack([jnp.cos(thetas), jnp.sin(thetas)], axis=-1)

# tangential angles (standard polar convention)
angles_raw = thetas + jnp.pi / 2
angles = wrap_to_pi(angles_raw)
x0 = start_point
p0 = jnp.stack([jnp.cos(angles_raw), jnp.sin(angles_raw)], axis=-1)


# Create rays fanning out in different directions
# start_point = jnp.array([0.5, 0.5])
# n_rays = 81  # Number of rays to shoot

# angles = jnp.linspace(-jnp.pi/6, jnp.pi/6, n_rays)  # small vertical spread

# angles = jnp.linspace(-jnp.pi / 3, jnp.pi / 3, n_rays)  # 60 degree fan
# x0 = jnp.tile(start_point, (n_rays, 1))  # All rays start from same point

# # Initial directions (unit vectors)
# p0 = jnp.stack([jnp.sin(angles), jnp.cos(angles)], axis=-1)

# Other initial conditions
mode = jnp.ones((n_rays, 1))
a0 = jnp.ones((n_rays, 1)) * 0.1
alpha0 = jnp.ones((n_rays, d)) * 1j
M0 = gb_utils.prepare_M0(alpha0, None)
lam = 0

# Time setup
ts = jnp.linspace(0, 1, 200)  # Longer time to see caustic formation

# Solver configuration
solver = gb_solvers.solve_ODE_base
solver_config = gb_solvers.SolverConfig(
    solver=diffrax.Tsit5(),
    max_steps=4096,
    rtol=1e-5,
    pcoeff=0.1,
    icoeff=0.3,
    dcoeff=0.0,
)

print("Running ray tracing simulation...")

# Run the simulation
xt, pt, mt, at = solver(x0, p0, M0, a0, mode, ts, c_maxwell_fisheye, lam, None)

print(f"Simulation complete. Ray trajectories shape: {xt.shape}")

# Set up color mapping for angles
angle_norm = mcolors.Normalize(vmin=-np.pi, vmax=np.pi)
colormap = cm.plasma

# Create the plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8))

# Plot 1: Velocity map
c_vals = jax.vmap(jax.vmap(c_maxwell_fisheye))(XY)
im1 = ax1.imshow(
    c_vals.T,
    extent=[0, xmax[0], 0, xmax[1]],
    origin="lower",
    cmap="viridis",
    aspect="auto",
)
ax1.set_title("$c(x)$")
ax1.set_xlabel("$x$")
ax1.set_ylabel("$y$")
plt.colorbar(im1, ax=ax1, label="Speed of sound")

# Plot 2: Ray paths and caustics with angle-based coloring
im2 = ax2.imshow(
    c_vals.T,
    extent=[0, xmax[0], 0, xmax[1]],
    origin="lower",
    cmap="viridis",
    alpha=0.3,  # Make background semi-transparent
    aspect="auto",
)

# Plot all ray trajectories with color based on angle
for i in range(n_rays):
    color = colormap(angle_norm(angles[i]))
    ax2.plot(xt[i, :, 0], xt[i, :, 1], color=color, linewidth=0.8, alpha=0.7)

ax2.scatter(start_point[:, 0], start_point[:, 1], color="black", s=10, zorder=3)
ax2.set_title("Ray Trajectories")
ax2.set_xlabel("$x$")
ax2.set_ylabel("$y$")
ax2.set_xlim(0, xmax[0])
ax2.set_ylim(0, xmax[1])

# Add colorbar for ray angles
sm = cm.ScalarMappable(cmap=colormap, norm=angle_norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax2, label="Ray Angle (radians)")
set_colorbar_pi_ticks(cbar, -np.pi, np.pi)

plt.tight_layout()
plt.savefig(PLOT_DIR / "trapping_rays.png", dpi=300, bbox_inches="tight")
plt.show()

######################
### 3D Visualization##
######################

# Create 3D caustic visualization (x, y, t) with colored rays

fig = plt.figure(figsize=(12, 10))
ax3d = fig.add_subplot(111, projection="3d")

# Plot ray trajectories in 3D (x, y, t) with angle-based coloring
time_array = ts
for i in range(n_rays):
    # Extract ray path
    x_path = xt[i, :, 0]
    y_path = xt[i, :, 1]

    # Get color for this ray based on its angle
    color = colormap(angle_norm(angles[i]))

    # Plot trajectory with time as z-axis
    ax3d.plot(x_path, y_path, time_array, color=color, linewidth=1, alpha=0.7)

ax3d.scatter(
    start_point[:, 0],
    start_point[:, 1],
    jnp.zeros_like(start_point[:, 0]),
    color="black",
    s=10,
    zorder=3,
)
# Set labels and title
ax3d.set_xlabel("$x$")
ax3d.set_ylabel("$y$")
ax3d.set_zlabel("$t$")
ax3d.set_title("Space-time Ray Trajectories")

# Set axis limits
ax3d.set_xlim(0, xmax[0])
ax3d.set_ylim(0, xmax[1])
ax3d.set_zlim(0, ts[-1])

# Add colorbar for ray angles
sm = cm.ScalarMappable(cmap=colormap, norm=angle_norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax3d, label="Ray Angle", shrink=0.8)
set_colorbar_pi_ticks(cbar, -np.pi, np.pi)

# Adjust viewing angle for better caustic visibility
ax3d.view_init(elev=20, azim=45)
plt.tight_layout()
plt.savefig(PLOT_DIR / "3d_trapping_rays.png", dpi=300, bbox_inches="tight")
plt.show()

# Create another 3D view focusing on caustic region with colored rays
fig = plt.figure(figsize=(12, 10))
ax3d_zoom = fig.add_subplot(111, projection="3d")

# Plot only the central region where caustics are likely to form
caustic_xlim = [0.3, 0.7]  # Focus on central region
caustic_ylim = [0.5, 1.5]  # Focus on where rays converge

for i in range(n_rays):
    x_path = xt[i, :, 0]
    y_path = xt[i, :, 1]

    # Only plot points within the caustic region
    mask = (
        (x_path >= caustic_xlim[0])
        & (x_path <= caustic_xlim[1])
        & (y_path >= caustic_ylim[0])
        & (y_path <= caustic_ylim[1])
    )

    if jnp.any(mask):
        # Get color for this ray based on its angle
        color = colormap(angle_norm(angles[i]))
        ax3d_zoom.plot(
            x_path[mask],
            y_path[mask],
            time_array[mask],
            color=color,
            linewidth=1.5,
            alpha=0.8,
        )

ax3d_zoom.set_xlabel("$x$")
ax3d_zoom.set_ylabel("$y$")
ax3d_zoom.set_zlabel("$t$")
ax3d_zoom.set_title("Zoomed in Space-time Ray Trajectories")

ax3d_zoom.set_xlim(caustic_xlim)
ax3d_zoom.set_ylim(caustic_ylim)
ax3d_zoom.set_zlim(0, ts[-1])

# Add colorbar for ray angles
sm = cm.ScalarMappable(cmap=colormap, norm=angle_norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax3d_zoom, label="Ray Angle", shrink=0.8)
set_colorbar_pi_ticks(cbar, -np.pi, np.pi)

# Better viewing angle for caustic detail
ax3d_zoom.view_init(elev=30, azim=60)

plt.tight_layout()
plt.savefig(PLOT_DIR / "3d_trapping_rays_zoom.png", dpi=300, bbox_inches="tight")
plt.show()

# Print some statistics
print(f"Velocity range: {c_vals.min():.3f} to {c_vals.max():.3f}")
print(f"Perturbation amplitude: {(c_vals.max() - c_vals.min()):.3f}")
print(
    f"Relative perturbation: {(c_vals.max() - c_vals.min()) / c_vals.mean() * 100:.1f}%"
)
print(f"Ray angle range: {angles.min():.3f} to {angles.max():.3f} radians")
print(
    f"Ray angle range: {jnp.degrees(angles.min()):.1f} to {jnp.degrees(angles.max()):.1f} degrees"
)

# 2d caustics

In [ ]:
# Choose which singularity to generate: "A2" (fold) or "A3" (cusp)
SCENARIO = "A3"

d = 2
lengths = 1.0
N = jnp.array([256, int(256 * lengths)])
xmax = jnp.array([1.0, lengths])
dx = xmax / N
x_linspace = [jnp.linspace(0.0, xmax[i], N[i]) for i in range(d)]
XY = jnp.stack(jnp.meshgrid(*x_linspace, indexing="ij"), axis=-1)


T_final = 1.10
nt = 450
ts = jnp.linspace(0.0, T_final, nt)


def n_luneburg_astig(x, center=jnp.array([0.5, 0.5]), Rx=0.33, Ry=0.28):
    dx = x[..., 0] - center[0]
    dy = x[..., 1] - center[1]
    val = 2.0 - (dx / Rx) ** 2 - (dy / Ry) ** 2
    # inside lens: n = sqrt(val); outside: n = 1
    n_core = jnp.sqrt(jnp.maximum(val, 1e-12))
    return jnp.where(val >= 1.0, n_core, 1.0)


def n_luneburg_astig_cubic(x, center=jnp.array([0.5, 0.5]), Rx=0.33, Ry=0.28, C=0.05):
    dx = x[..., 0] - center[0]
    dy = x[..., 1] - center[1]
    val = 2.0 - (dx / Rx) ** 2 - (dy / Ry) ** 2
    n_core = jnp.sqrt(jnp.maximum(val, 1e-12))
    # normalize coords to lens scale to keep the cubic term O(1)
    ux, uy = dx / Rx, dy / Ry
    cubic = ux**3 - 3.0 * ux * uy**2  # Re[(ux+iuy)^3]
    # apply cubic only where lens exists (val >= 1)
    aberr = 1.0 + C * cubic
    # prevent nonphysical n by clipping low tail
    n_core_ab = jnp.clip(n_core * aberr, 0.4, None)
    n = jnp.where(val >= 1.0, n_core_ab, 1.0)
    return n


def c_from_n(n_fn, c0=1.0, **kwargs):
    def c_fn(x):
        return c0 / n_fn(x, **kwargs)

    return c_fn


# Select medium
if SCENARIO == "A2":
    c_fn = c_from_n(n_luneburg_astig, c0=1.0, Rx=0.33, Ry=0.28)
    medium_name = "A2_fold_astig_luneburg"
elif SCENARIO == "A3":
    c_fn = c_from_n(n_luneburg_astig_cubic, c0=1.0, Rx=0.33, Ry=0.28, C=0.06)
    medium_name = "A3_cusp_astig_luneburg_cubic"
else:
    raise ValueError("SCENARIO must be 'A2' or 'A3'.")

# --------- initial data: plane wave entering lens ----------
# Start a line of points near the bottom edge, moving straight up.
n_rays = 121
x_line = jnp.linspace(0.15, 0.85, n_rays)
y0 = jnp.full_like(x_line, 0.10)
x0 = jnp.stack([x_line, y0], axis=-1)  # [n_rays, 2]
p0 = jnp.tile(jnp.array([0.0, 1.0]), (n_rays, 1))  # same direction (plane wave)

# GB ancillary data (not used for topology; keep consistent with your solver)
mode = jnp.ones((n_rays, 1))
a0 = jnp.ones((n_rays, 1)) * 0.1
alpha0 = jnp.ones((n_rays, d)) * 1j
M0 = gb_utils.prepare_M0(alpha0, None)
lam = 0.0

# --------- solver config ----------
solver = gb_solvers.solve_ODE_base
solver_config = gb_solvers.SolverConfig(
    solver=diffrax.Tsit5(),
    max_steps=8192,
    rtol=1e-5,
    pcoeff=0.1,
    icoeff=0.3,
    dcoeff=0.0,
)

print(f"Running rays for {SCENARIO} …")
xt, pt, mt, at = solver(x0, p0, M0, a0, mode, ts, c_fn, lam, None)
print("Done. xt shape:", xt.shape)  # [n_rays, nt, 2]


# --------- caustic detector: J = det [∂x/∂α, ∂x/∂t] ----------
def caustic_indicator(xt, ts):
    X = np.asarray(xt)  # [n_rays, nt, 2]
    T = np.asarray(ts)  # [nt]
    # finite differences in the ray label (α: here the ray index) and time
    dX_dalpha = 0.5 * (X[2:, :, :] - X[:-2, :, :])  # [n_rays-2, nt, 2]
    dX_dt = (X[1:-1, 1:, :] - X[1:-1, :-1, :]) / (
        T[1:, None] - T[:-1, None]
    )  # [n_rays-2, nt-1, 2]
    # align shapes
    dX_dalpha = dX_dalpha[:, :-1, :]  # [n_rays-2, nt-1, 2]
    # 2x2 determinant of [dX/dalpha, dX/dt]
    J = dX_dalpha[..., 0] * dX_dt[..., 1] - dX_dalpha[..., 1] * dX_dt[..., 0]
    return J  # near zero ⇒ caustic


J = caustic_indicator(xt, ts)  # [n_rays-2, nt-1]
absJ = np.abs(J)
# threshold by a small quantile to pick points near the caustic
q = np.quantile(absJ, 0.015)
mask = absJ <= q

# Map those (i,t) points to (x,y)
X_mid = np.asarray(xt)[1:-1, :-1, :]  # [n_rays-2, nt-1, 2]
caustic_xy = X_mid[mask]  # [K, 2]

# --------- background medium image ----------
c_vals = jax.vmap(jax.vmap(c_fn))(XY)
cmin = float(c_vals.min())
cmax = float(c_vals.max())

# --------- plotting ----------
# Color by initial x-position
norm = mcolors.Normalize(vmin=float(x_line.min()), vmax=float(x_line.max()))
cmap = cm.plasma


def compute_ray_angles(p0=None, xt=None):
    """
    Return per-ray angles in radians.
    Prefer p0; otherwise estimate from the earliest displacement in xt.
    """
    if p0 is not None:
        v = jnp.asarray(p0)  # [n_rays, 2]
    elif (xt is not None) and (xt.shape[1] >= 2):
        # first-step velocity estimate
        v = jnp.asarray(xt[:, 1, :] - xt[:, 0, :])  # [n_rays, 2]
        # handle any zero-length steps by using a slightly later time
        norms = jnp.linalg.norm(v, axis=1)
        if bool(jnp.any(norms == 0)):
            k = min(5, int(xt.shape[1]) - 1)
            v_alt = jnp.asarray(xt[:, k, :] - xt[:, 0, :])
            v = jnp.where(norms[:, None] == 0, v_alt, v)
    else:
        raise ValueError(
            "Cannot infer angles: provide p0 or xt with at least 2 time steps."
        )
    return jnp.arctan2(v[:, 1], v[:, 0])  # [-pi, pi]


# If 'angles' is not already defined, make it now.
try:
    angles  # noqa
except NameError:
    angles_raw = compute_ray_angles(p0=p0 if "p0" in locals() else None, xt=xt)
    angles = wrap_to_pi(angles_raw)


# =========================
# Caustic indicator helpers
# =========================
def _jacobian_dets(xt, ts):
    """
    Compute J = det[ ∂x/∂α , ∂x/∂t ] with α = ray index.
    xt: [n_rays, nt, 2], ts: [nt]
    Returns:
      J: [n_rays-2, nt-1]
      X_mid: [n_rays-2, nt-1, 2] (aligned with J)
    """
    X = jnp.asarray(xt)
    T = jnp.asarray(ts)
    dX_dalpha = 0.5 * (X[2:, :, :] - X[:-2, :, :])  # [R-2, T, 2]
    dT = (T[1:] - T[:-1]).reshape(1, -1, 1)  # [1, T-1, 1]
    dX_dt = (X[1:-1, 1:, :] - X[1:-1, :-1, :]) / dT  # [R-2, T-1, 2]
    dX_dalpha = dX_dalpha[:, :-1, :]  # [R-2, T-1, 2]
    J = (
        dX_dalpha[..., 0] * dX_dt[..., 1] - dX_dalpha[..., 1] * dX_dt[..., 0]
    )  # [R-2, T-1]
    return J, X[1:-1, :-1, :]


def _caustic_cloud(J, X_mid, quantile=0.01):
    """Points where |J| is very small."""
    absJ = jnp.abs(J)
    q = jnp.quantile(absJ, quantile)
    mask = absJ <= q
    cloud_xy = jnp.asarray(X_mid)[mask]
    return jnp.asarray(cloud_xy), jnp.asarray(mask)


def _caustic_curve_alpha(J, X_mid, ts):
    """Crisp J=0 curve via sign changes along α for each t."""
    Jnp = np.asarray(J)
    Xnp = np.asarray(X_mid)
    Tnp = np.asarray(ts[:-1])
    sgn = np.sign(Jnp)
    sgn[sgn == 0] = 1
    xs, ys, zs = [], [], []
    for j in range(Jnp.shape[1]):  # time
        s = sgn[:, j]
        cross = np.where(s[:-1] * s[1:] < 0)[0]
        for i in cross:
            J1, J2 = Jnp[i, j], Jnp[i + 1, j]
            s_alpha = -J1 / (J2 - J1) if J2 != J1 else 0.5
            s_alpha = float(np.clip(s_alpha, 0.0, 1.0))
            P = (1.0 - s_alpha) * Xnp[i, j, :] + s_alpha * Xnp[i + 1, j, :]
            xs.append(P[0])
            ys.append(P[1])
            zs.append(Tnp[j])
    if len(xs) == 0:
        return np.empty((0,)), np.empty((0,)), np.empty((0,))
    return np.array(xs), np.array(ys), np.array(zs)


# Compute caustic diagnostics
J, X_mid = _jacobian_dets(xt, ts)  # J: [n_rays-2, nt-1]
cloud_xy, cloud_mask = _caustic_cloud(J, X_mid)  # small-|J| cloud
cx, cy, cz = _caustic_curve_alpha(J, X_mid, ts)  # crisp J=0 curve


# Set up color mapping for angles
angle_norm = mcolors.Normalize(vmin=float(angles.min()), vmax=float(angles.max()))
colormap = cm.plasma

# Create the plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8))

# Plot 1: Velocity map
c_vals = jax.vmap(jax.vmap(c_fn))(XY)
im1 = ax1.imshow(
    c_vals.T,
    extent=[0, xmax[0], 0, xmax[1]],
    origin="lower",
    cmap="viridis",
    aspect="auto",
)
ax1.set_title("$c(x)$")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
plt.colorbar(im1, ax=ax1, label="Speed of sound")

# Plot 2: Ray paths and caustics with angle-based coloring
im2 = ax2.imshow(
    c_vals.T,
    extent=[0, xmax[0], 0, xmax[1]],
    origin="lower",
    cmap="viridis",
    alpha=0.3,  # semi-transparent background
    aspect="auto",
)

# Plot all ray trajectories with color based on angle
for i in range(n_rays):
    color = colormap(angle_norm(float(angles[i])))
    ax2.plot(xt[i, :, 0], xt[i, :, 1], color=color, linewidth=0.8, alpha=0.7)

# Caustic overlays
if cloud_xy.size > 0:
    ax2.scatter(
        np.asarray(cloud_xy[:, 0]),
        np.asarray(cloud_xy[:, 1]),
        s=6,
        c="crimson",
        alpha=0.4,
        label="caustic cloud (|J| small)",
    )
if cx.size > 0:
    ax2.scatter(cx, cy, s=14, c="black", alpha=0.95, label="caustic (J=0)")

ax3d.scatter(
    start_point[:, 0],
    start_point[:, 1],
    jnp.zeros_like(start_point[:, 0]),
    color="black",
    s=20,
    zorder=3,
)

ax2.set_title("Ray Trajectories and Caustics")
ax2.set_xlabel("$x$")
ax2.set_ylabel("$y$")
ax2.set_xlim(0, xmax[0])
ax2.set_ylim(0, xmax[1])

# Add colorbar for ray angles
sm = cm.ScalarMappable(cmap=colormap, norm=angle_norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax2, label="Ray Angle")
set_colorbar_pi_ticks(cbar, -np.pi, np.pi)

if (cloud_xy.size > 0) or (cx.size > 0):
    ax2.legend(loc="upper right", frameon=False)

plt.tight_layout()
plt.savefig(
    PLOT_DIR / f"rays_and_velocity_maps_{SCENARIO}.png", dpi=300, bbox_inches="tight"
)
plt.show()

######################
### 3D Visualization##
######################

fig = plt.figure(figsize=(12, 10))
ax3d = fig.add_subplot(111, projection="3d")

# Plot ray trajectories in 3D (x, y, t) with angle-based coloring
time_array = ts
for i in range(n_rays):
    x_path = xt[i, :, 0]
    y_path = xt[i, :, 1]
    color = colormap(angle_norm(float(angles[i])))
    ax3d.plot(x_path, y_path, time_array, color=color, linewidth=1, alpha=0.7)

# Caustic in 3D
if cx.size > 0:
    ax3d.scatter(
        cx, cy, cz, s=16, c="black", alpha=0.95, depthshade=False, label="caustic (J=0)"
    )
if cloud_xy.size > 0:
    jj = np.where(np.asarray(cloud_mask))[1]  # time indices for cloud points
    tz = np.asarray(ts[:-1])[jj]
    ax3d.scatter(
        np.asarray(cloud_xy[:, 0]),
        np.asarray(cloud_xy[:, 1]),
        tz,
        s=8,
        c="crimson",
        alpha=0.35,
    )

ax3d.set_xlabel("$x$")
ax3d.set_ylabel("$y$")
ax3d.set_zlabel("$t$")
ax3d.set_title("Space-time Ray Trajectories")
ax3d.set_xlim(0, xmax[0])
ax3d.set_ylim(0, xmax[1])
ax3d.set_zlim(0, ts[-1])

sm = cm.ScalarMappable(cmap=colormap, norm=angle_norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax3d, label="Ray Angle", shrink=0.8)
set_colorbar_pi_ticks(cbar, -np.pi, np.pi)

ax3d.view_init(elev=20, azim=45)
plt.tight_layout()
plt.savefig(PLOT_DIR / f"3d_caustic_{SCENARIO}.png", dpi=300, bbox_inches="tight")
plt.show()

# 3D zoom view (optional, unchanged except caustic overlay filtered by window)
fig = plt.figure(figsize=(12, 10))
ax3d_zoom = fig.add_subplot(111, projection="3d")

caustic_xlim = [0.3, 0.7]
caustic_ylim = [0.5, 1.5]

for i in range(n_rays):
    x_path = xt[i, :, 0]
    y_path = xt[i, :, 1]
    mask = (
        (x_path >= caustic_xlim[0])
        & (x_path <= caustic_xlim[1])
        & (y_path >= caustic_ylim[0])
        & (y_path <= caustic_ylim[1])
    )
    if jnp.any(mask):
        color = colormap(angle_norm(float(angles[i])))
        ax3d_zoom.plot(
            x_path[mask],
            y_path[mask],
            time_array[mask],
            color=color,
            linewidth=1.5,
            alpha=0.8,
        )

# Caustic points inside the zoom box
if cx.size > 0:
    mask_zoom = (
        (cx >= caustic_xlim[0])
        & (cx <= caustic_xlim[1])
        & (cy >= caustic_ylim[0])
        & (cy <= caustic_ylim[1])
    )
    if np.any(mask_zoom):
        ax3d_zoom.scatter(
            cx[mask_zoom],
            cy[mask_zoom],
            cz[mask_zoom],
            s=20,
            c="black",
            alpha=0.95,
            depthshade=False,
            label="caustic (J=0)",
        )

ax3d_zoom.set_xlabel("$x$")
ax3d_zoom.set_ylabel("$y$")
ax3d_zoom.set_zlabel("$t$")
ax3d_zoom.set_title("Zoomed Space time Ray Trajectories")
ax3d_zoom.set_xlim(caustic_xlim)
ax3d_zoom.set_ylim(caustic_ylim)
ax3d_zoom.set_zlim(0, ts[-1])

sm = cm.ScalarMappable(cmap=colormap, norm=angle_norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax3d_zoom, label="Ray Angle", shrink=0.8)
set_colorbar_pi_ticks(cbar, -np.pi, np.pi)

ax3d_zoom.view_init(elev=30, azim=60)
plt.tight_layout()
plt.savefig(
    PLOT_DIR / f"3d_caustic_formation_{SCENARIO}.png", dpi=300, bbox_inches="tight"
)
plt.show()

# Print some statistics
print(f"Caustic cloud points: {int(cloud_xy.shape[0])}")
print(f"Caustic curve points: {int(cx.size)}")

# 3D Caustics

In [ ]:
#!/usr/bin/env python
# coding: utf-8

import jax
import jax.numpy as jnp
import numpy as np
import diffrax
import matplotlib.pyplot as plt
from matplotlib import cm, colors as mcolors
from pathlib import Path

from beamax.gb import gb_utils, gb_solvers  # your package
from beamax.plotter import use_beamax_style

ROOT_DIR = utils.detect_root()
PLOT_DIR = ROOT_DIR / "plots"
DATA_DIR = ROOT_DIR / "data"
PROF_DIR = ROOT_DIR / "profiler"
PLOT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)
PROF_DIR.mkdir(exist_ok=True)
use_beamax_style()

jax.config.update("jax_enable_x64", True)

# ===================== 3D medium (astigmatic Luneburg + cubic) =================


def n_luneburg3d_astig(x, center=jnp.array([0.5, 0.5, 0.5]), Rx=0.33, Ry=0.28, Rz=0.36):
    """3D astigmatic Luneburg refractive index, n=√(2 - (dx/Rx)^2 - ...), clipped to n>=1 outside lens."""
    dx = x[..., 0] - center[0]
    dy = x[..., 1] - center[1]
    dz = x[..., 2] - center[2]
    val = 2.0 - (dx / Rx) ** 2 - (dy / Ry) ** 2 - (dz / Rz) ** 2
    n_core = jnp.sqrt(jnp.maximum(val, 1e-12))
    return jnp.where(val >= 1.0, n_core, 1.0)


def n_luneburg3d_astig_cubic(
    x, center=jnp.array([0.5, 0.5, 0.5]), Rx=0.33, Ry=0.28, Rz=0.36, C=0.06
):
    """Add weak cubic aberration in x,y to create cusp edges on the fold surface."""
    dx = x[..., 0] - center[0]
    dy = x[..., 1] - center[1]
    dz = x[..., 2] - center[2]
    val = 2.0 - (dx / Rx) ** 2 - (dy / Ry) ** 2 - (dz / Rz) ** 2
    n_core = jnp.sqrt(jnp.maximum(val, 1e-12))
    ux, uy = dx / Rx, dy / Ry
    cubic = ux**3 - 3.0 * ux * uy**2  # Re[(ux+i uy)^3]
    aberr = 1.0 + C * cubic
    n_mod = jnp.clip(n_core * aberr, 0.4, None)
    return jnp.where(val >= 1.0, n_mod, 1.0)


def c_from_n(n_fn, c0=1.0, **kwargs):
    def c_fn(x):
        return c0 / n_fn(x, **kwargs)

    return c_fn


# Choose medium
MEDIUM = "A2"  # "A2" for fold-only, "A3" for cusp edges
if MEDIUM == "A2":
    c_fn = c_from_n(n_luneburg3d_astig, Rx=0.33, Ry=0.28, Rz=0.36)
    medium_name = "A2_fold_Luneburg3D"
elif MEDIUM == "A3":
    c_fn = c_from_n(n_luneburg3d_astig_cubic, Rx=0.33, Ry=0.28, Rz=0.36, C=0.06)
    medium_name = "A3_cusp_Luneburg3D_cubic"
else:
    raise ValueError("MEDIUM must be 'A2' or 'A3'.")

# ======================= Launch manifold and times =============================

# Two-parameter source patch on plane z = z0, with normal incidence (+z)
Na, Nb = 41, 41  # parameters α (x), β (y); keep modest to avoid RAM blowups
x_min, x_max = 0.20, 0.80
y_min, y_max = 0.20, 0.80
z0 = 0.08

alpha_grid = jnp.linspace(x_min, x_max, Na)  # α → x
beta_grid = jnp.linspace(y_min, y_max, Nb)  # β → y
AA, BB = jnp.meshgrid(alpha_grid, beta_grid, indexing="ij")  # [Na,Nb]
X0 = jnp.stack([AA, BB, jnp.full_like(AA, z0)], axis=-1)  # [Na,Nb,3]
x0 = X0.reshape(-1, 3)  # [Na*Nb,3]

# Initial directions (plane wave entering along +z)
p0 = jnp.tile(jnp.array([0.0, 0.0, 1.0]), (Na * Nb, 1))

# GB ancillary (not used for topology but required by your solver)
mode = jnp.ones((Na * Nb, 1))
a0 = jnp.ones((Na * Nb, 1)) * 0.1
alpha0 = jnp.ones((Na * Nb, 3)) * 1j
M0 = gb_utils.prepare_M0(alpha0, None)
lam = 0.0

# Time grid: midpoints (k+1/2) align with our J computation below
T_final = 1.15
Nt = 280
ts = jnp.linspace(0.0, T_final, Nt)

# =========================== Integrate rays ===================================

solver = gb_solvers.solve_ODE_base
solver_config = gb_solvers.SolverConfig(
    solver=diffrax.Tsit5(),
    max_steps=12000,
    rtol=1e-5,
    pcoeff=0.1,
    icoeff=0.3,
    dcoeff=0.0,
)
print(f"[{medium_name}] integrating {Na}x{Nb} rays over {Nt} steps …")
xt, pt, mt, at = solver(x0, p0, M0, a0, mode, ts, c_fn, lam, None)  # xt: [Na*Nb,Nt,3]
Xt = np.asarray(xt).reshape(Na, Nb, Nt, 3)  # numpy for downstream ops

# ======================= Derivatives and J(α,β,t) =============================

# central diffs in α,β; forward in t; align to common interior grid:
#   α ∈ [1..Na-2], β ∈ [1..Nb-2], t ∈ [0..Nt-2]  (J has shape [Na-2, Nb-2, Nt-1])
dX_dalpha = 0.5 * (Xt[2:, 1:-1, :, :] - Xt[:-2, 1:-1, :, :])  # [Na-2, Nb-2, Nt, 3]
dX_dbeta = 0.5 * (Xt[1:-1, 2:, :, :] - Xt[1:-1, :-2, :, :])  # [Na-2, Nb-2, Nt, 3]
dX_dt = (Xt[1:-1, 1:-1, 1:, :] - Xt[1:-1, 1:-1, :-1, :]) / float(
    ts[1] - ts[0]
)  # [Na-2, Nb-2, Nt-1, 3]
# crop α,β derivatives in time to match dX_dt
dX_dalpha = dX_dalpha[:, :, :-1, :]
dX_dbeta = dX_dbeta[:, :, :-1, :]

# J = det[ dα x, dβ x, dt x ] at the aligned grid
# Construct 3x3 matrices per grid point
A = np.stack(
    [dX_dalpha, dX_dbeta, dX_dt], axis=-2
)  # [Na-2, Nb-2, Nt-1, 3(vec), 3(cols)]
# Determinant per point
J = np.linalg.det(A)  # [Na-2, Nb-2, Nt-1]

# ===================== Extract 3D caustic surface (point cloud) ===============

# For each time index k (corresponds to mid-time t_{k+1/2}), find zero crossings of J(α,β, k)
# along α and β edges, linearly interpolate to (α*,β*), then push to (x,y,z) at mid-time.


def push_to_space(alpha_star, beta_star, k):
    """
    Map fractional (α*,β*,t_mid at k+0.5) to (x,y,z) by tri-linear interpolation on Xt.
    α*,β* are in Xt index space (0..Na-1, 0..Nb-1). Time uses k and k+1 with 0.5 weight.
    """
    # neighbors in α,β
    ia0 = int(np.floor(alpha_star))
    ia1 = min(ia0 + 1, Na - 1)
    ib0 = int(np.floor(beta_star))
    ib1 = min(ib0 + 1, Nb - 1)
    fa = float(alpha_star - ia0)
    fb = float(beta_star - ib0)
    # time mid between k and k+1
    t0, t1 = k, k + 1

    # tri-linear weights (α,β,t)
    def w(a, b, t):
        return (
            ((1 - fa) if a == 0 else fa)
            * ((1 - fb) if b == 0 else fb)
            * (0.5 if t == 0 else 0.5)
        )

    X000 = Xt[ia0, ib0, t0, :]
    X001 = Xt[ia0, ib0, t1, :]
    X010 = Xt[ia0, ib1, t0, :]
    X011 = Xt[ia0, ib1, t1, :]
    X100 = Xt[ia1, ib0, t0, :]
    X101 = Xt[ia1, ib0, t1, :]
    X110 = Xt[ia1, ib1, t0, :]
    X111 = Xt[ia1, ib1, t1, :]
    P = (
        w(0, 0, 0) * X000
        + w(0, 0, 1) * X001
        + w(0, 1, 0) * X010
        + w(0, 1, 1) * X011
        + w(1, 0, 0) * X100
        + w(1, 0, 1) * X101
        + w(1, 1, 0) * X110
        + w(1, 1, 1) * X111
    )
    return P  # (3,)


def zero_cross_points_2d(Jk):
    """
    Zero-crossings of a 2D scalar field Jk[i,j] on the interior grid i∈[0..Na-3], j∈[0..Nb-3].
    Returns edge-intersection points in (i_frac, j_frac), each ∈ R^2 with fractional indices.
    """
    pts = []
    # α-edges
    for i in range(Jk.shape[0] - 1):
        J0 = Jk[i, :]
        J1 = Jk[i + 1, :]
        s = J0 * J1
        flips = np.where(s < 0)[0]  # indices j where sign flips between i and i+1
        for j in flips:
            t0, t1 = J0[j], J1[j]
            s_alpha = float(-t0 / (t1 - t0)) if t1 != t0 else 0.5
            pts.append((i + s_alpha, float(j)))
    # β-edges
    for j in range(Jk.shape[1] - 1):
        J0 = Jk[:, j]
        J1 = Jk[:, j + 1]
        s = J0 * J1
        flips = np.where(s < 0)[0]  # indices i where sign flips between j and j+1
        for i in flips:
            t0, t1 = J0[i], J1[i]
            s_beta = float(-t0 / (t1 - t0)) if t1 != t0 else 0.5
            pts.append((float(i), j + s_beta))
    return pts


# Map J-grid indices (i,j,k) to Xt indices:
# J is defined on α∈[1..Na-2], β∈[1..Nb-2], t∈[0..Nt-2].
# A zero-cross at (i_frac, j_frac) in J-index space corresponds to Xt α*=1+i_frac, β*=1+j_frac.
caustic_xyz = []
cusp_xyz = []


# Heuristic cusp flag: take the 3x3 Jacobian at each (i,j,k); at J≈0, compute its smallest
# singular vector (right-null dir); check |∇J·dir_par| small, and second dir-derivative not tiny.
def cusp_flag(i, j, k, J3x3, Jgrid):
    # J3x3: [3,3] = [dαx, dβx, dtx]
    try:
        u, s, vh = np.linalg.svd(J3x3, full_matrices=True)
    except np.linalg.LinAlgError:
        return False
    dir_null = vh[-1, :]  # right-singular vector with smallest σ (≈ null direction)

    # finite differences of scalar J in (i,j,k) param-time cube
    # handle bounds cautiously
    def safe(Jg, idx, axis):
        lo = max(idx - 1, 0)
        hi = min(idx + 1, Jg.shape[axis] - 1)
        return lo, hi

    ii0, ii1 = safe(Jgrid, i, axis=0)
    jj0, jj1 = safe(Jgrid, j, axis=1)
    kk0, kk1 = safe(Jgrid, k, axis=2)
    # central differences (scaled; absolute scale irrelevant for zero test)
    dJ_di = 0.5 * (
        Jgrid[min(i + 1, Jgrid.shape[0] - 1), j, k] - Jgrid[max(i - 1, 0), j, k]
    )
    dJ_dj = 0.5 * (
        Jgrid[i, min(j + 1, Jgrid.shape[1] - 1), k] - Jgrid[i, max(j - 1, 0), k]
    )
    dJ_dk = 0.5 * (
        Jgrid[i, j, min(k + 1, Jgrid.shape[2] - 1)] - Jgrid[i, j, max(k - 1, 0)]
    )
    g = np.array([dJ_di, dJ_dj, dJ_dk], dtype=float)
    # directional first derivative along param-time null direction
    first = abs(np.dot(g, dir_null))
    # crude second derivative (projected) via 1D second-diff stencil along dominant axis
    ax = int(np.argmax(np.abs(dir_null)))
    if ax == 0 and 1 <= i <= Jgrid.shape[0] - 2:
        second = Jgrid[i + 1, j, k] - 2 * Jgrid[i, j, k] + Jgrid[i - 1, j, k]
    elif ax == 1 and 1 <= j <= Jgrid.shape[1] - 2:
        second = Jgrid[i, j + 1, k] - 2 * Jgrid[i, j, k] + Jgrid[i, j - 1, k]
    elif ax == 2 and 1 <= k <= Jgrid.shape[2] - 2:
        second = Jgrid[i, j, k + 1] - 2 * Jgrid[i, j, k] + Jgrid[i, j, k - 1]
    else:
        second = 0.0
    # thresholds by robust quantiles
    # compute once outside? here we use simple relative thresholds
    return (first < 1e-3) and (abs(second) > 1e-6)


print("[extract] building caustic point cloud …")
for k in range(
    J.shape[2]
):  # time index for J: 0..Nt-2 (represents mid-time between ts[k] and ts[k+1])
    Jk = J[:, :, k]  # [Na-2, Nb-2]
    pts_ij = zero_cross_points_2d(Jk)
    if not pts_ij:
        continue
    for i_frac, j_frac in pts_ij:
        # map J indices to Xt indices (shift by +1 to undo central-diff contraction)
        a_star = 1.0 + i_frac
        b_star = 1.0 + j_frac
        # push to space at time mid (k+1/2)
        P = push_to_space(a_star, b_star, k)
        caustic_xyz.append(P)
        # optional cusp flag at nearest integer J-grid point
        ii = int(round(i_frac))
        jj = int(round(j_frac))
        # reconstruct local 3x3 Jacobian from stored derivatives
        dα = dX_dalpha[ii, jj, k, :]  # aligned with J indices
        dβ = dX_dbeta[ii, jj, k, :]
        dt = dX_dt[ii, jj, k, :]
        J3 = np.stack([dα, dβ, dt], axis=-1)
        if cusp_flag(ii, jj, k, J3, J):
            cusp_xyz.append(P)

caustic_xyz = np.array(caustic_xyz) if len(caustic_xyz) else np.empty((0, 3))
cusp_xyz = np.array(cusp_xyz) if len(cusp_xyz) else np.empty((0, 3))

# ============================== Diagnostics ===================================

print(f"[{medium_name}] rays: {Na}x{Nb}, Nt={Nt}")
print(f"Caustic points (cloud): {len(caustic_xyz)}")
if len(caustic_xyz):
    mn = caustic_xyz.min(axis=0)
    mx = caustic_xyz.max(axis=0)
    print(
        f"Space bounds of caustic cloud: x∈[{mn[0]:.3f},{mx[0]:.3f}] "
        f"y∈[{mn[1]:.3f},{mx[1]:.3f}] z∈[{mn[2]:.3f},{mx[2]:.3f}]"
    )
print(f"Cusp-edge candidates: {len(cusp_xyz)}")

# ============================== Plotting ======================================

# Subsample rays for clarity
skip_a = max(1, Na // 12)
skip_b = max(1, Nb // 12)
idx_a = np.arange(0, Na, skip_a)
idx_b = np.arange(0, Nb, skip_b)

# color by launch x (α)
norm = mcolors.Normalize(vmin=float(alpha_grid.min()), vmax=float(alpha_grid.max()))
cmap = cm.plasma

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection="3d")
# rays
for ia in idx_a:
    for ib in idx_b:
        traj = Xt[ia, ib, :, :]  # [Nt,3]
        ax.plot(
            traj[:, 0],
            traj[:, 1],
            traj[:, 2],
            lw=0.7,
            alpha=0.5,
            color=cmap(norm(float(alpha_grid[ia]))),
        )
# caustic surface (point cloud)
if len(caustic_xyz):
    ax.scatter(
        caustic_xyz[:, 0],
        caustic_xyz[:, 1],
        caustic_xyz[:, 2],
        s=4,
        c="black",
        alpha=0.9,
        depthshade=False,
        label="fold (J=0)",
    )
# cusp-edge markers
if len(cusp_xyz):
    ax.scatter(
        cusp_xyz[:, 0],
        cusp_xyz[:, 1],
        cusp_xyz[:, 2],
        s=16,
        c="crimson",
        alpha=0.9,
        depthshade=False,
        label="cusp edge (A3)",
    )

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_zlim(0, 1)
ax.set_title(f"3D Spatial Caustics — {medium_name}")
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label="launch α (x on source patch)", shrink=0.8)
if len(caustic_xyz) or len(cusp_xyz):
    ax.legend(loc="upper right", frameon=False)
ax.view_init(elev=22, azim=48)
plt.tight_layout()
plt.savefig(
    PLOT_DIR / f"caustic_surface_pointcloud_{medium_name}.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

# ============== Optional: per-time counts (list "3D caustics" over time) ======

counts_per_t = []
for k in range(J.shape[2]):
    Jk = J[:, :, k]
    c = np.count_nonzero(Jk[:-1, :] * Jk[1:, :] < 0) + np.count_nonzero(
        Jk[:, :-1] * Jk[:, 1:] < 0
    )
    counts_per_t.append(c)
print("Per-time zero-cross counts (proxy for surface complexity):")
print(counts_per_t[:20], "... total:", sum(counts_per_t))